# **#TelegramScrap: A comprehensive tool for scraping Telegram data**

✅ This code, developed by [Ergon Cugler de Moraes Silva](https://github.com/ergoncugler) (Brazil), aims to scrape data from selected `Telegram Channels, Groups, or Chats` using the `Telethon Library`. It is designed to facilitate the extraction of various data fields including `message content, author information, reactions, views, and comments`. The primary functions of this code include setting up scraping parameters, processing messages and their associated comments, and handling unsupported characters to ensure data integrity. Data is stored in `Apache Parquet files (.parquet)`, which are highly efficient for both storage and processing, making them superior to traditional spreadsheets in terms of speed and scalability. This tool is particularly **useful for researchers and analysts** looking to collect and analyze Telegram data efficiently.

✅ **The code is open-source and available for free at [https://github.com/ergoncugler/web-scraping-telegram/](https://github.com/ergoncugler/web-scraping-telegram/)**. While it is free to use and modify, the responsibility for its use and any modifications lies with the user. Feel free to explore, utilize, and adapt the code to suit your needs, but please ensure you comply with Telegram's terms of service and data privacy regulations.

✅ Once the data is extracted into `.parquet` files, various authoring tools are available for analyzing this data. Examples include: [**combine_scraped_parquet_files.py**](https://github.com/ergoncugler/web-scraping-telegram/blob/main/combine_scraped_parquet_files.py), which combines multiple Parquet files into a single DataFrame, removing duplicates and adjusting columns; [**generate_groups_month_summary.py**](https://github.com/ergoncugler/web-scraping-telegram/blob/main/generate_groups_month_summary.py), which creates monthly summary tables for each group, showing the number of contents and comments; [**sample_data_from_parquet_to_excel.py**](https://github.com/ergoncugler/web-scraping-telegram/blob/main/sample_data_from_parquet_to_excel.py), which samples data proportionally based on categories and saves it to an Excel file; [**scrape_and_filter_by_keywords_from_parquet_to_excel.py**](https://github.com/ergoncugler/web-scraping-telegram/blob/main/scrape_and_filter_by_keywords_from_parquet_to_excel.py), which filters rows based on keywords, adds indicator columns for each keyword, and saves the results to Excel files; and [**snowballing_scrape_telegram_links_from_data.py**](https://github.com/ergoncugler/web-scraping-telegram/blob/main/snowballing_scrape_telegram_links_from_data.py), which extracts, normalizes, and counts Telegram links, saving the analysis to an Excel file.

## **#Papers: Some scientific production using this code**

✅ In the realm of scientific articles, this code was instrumental in the study **Informational Co-option against Democracy: Comparing Bolsonaro's Discourses about Voting Machines with the Public Debate** ([**link**](https://dl.acm.org/doi/abs/10.1145/3614321.3614373)). It was also used in **Institutional Denialism From the President's Speeches to the Formation of the Early Treatment Agenda (Off Label) in the COVID-19 Pandemic in Brazil** ([**link**](https://anepecp.org/ojs/index.php/br/article/view/561)). Moreover, the code facilitated research in **Catalytic Conspiracism: Exploring Persistent Homologies Time Series in the Dissemination of Disinformation in Conspiracy Theory Communities on Telegram** ([**link**](https://www.abcp2024.sinteseeventos.com.br/trabalho/view?ID_TRABALHO=687)) and **Conspiratorial Convergence: Comparing Thematic Agendas Among Conspiracy Theory Communities on Telegram Using Topic Modeling** ([**link**](https://www.abcp2024.sinteseeventos.com.br/trabalho/view?ID_TRABALHO=903)). Lastly, it was pivotal in the study **Informational Disorder and Institutions Under Attack: How Did Former President Bolsonaro's Narratives Against the Brazilian Judiciary Between 2019 and 2022 Manifest?** ([**link**](https://www.encontro2023.anpocs.org.br/trabalho/view?ID_TRABALHO=8990)).

✅ Furthermore, the code was utilized in several technical notes, as we can see in **Technical Note #16 – Disinformation about Electronic Voting Machines Persists Outside Election Periods** ([**link**](https://www.monitordigital.org/2023/05/18/nota-tecnica-16-desinformacao-sobre-urnas-eletronicas-persiste-fora-dos-periodos-eleitorais/)). It was also employed in **Technical Note #18 – Electoral Fraud Discourse in Argentina on Telegram and Twitter** ([**link**](https://www.monitordigital.org/2023/10/24/nota-tecnica-18-discurso-de-fraude-eleitoral-na-argentina-no-telegram-e-no-twitter/)). The code contributed to the analysis in the technical note **Bashing and Praising Public Servants and Bureaucrats During the Bolsonaro Government (2019 - 2022)** ([**link**](https://neburocracia.wordpress.com/wp-content/uploads/2024/04/nota-tecnica-neb-fgv-eaesp-como-bolsonaro-equilibrou-ataques-e-acenos-aos-servidores-publicos-e-burocratas-entre-2019-e-2022.pdf)). Additionally, it was used in **Technical Note 2: The Digital Territory of Milei's Followers: From Commerce to Politics** ([**link**](https://pacunla.com/nota-tecnica-2-el-territorio-digital-de-los-seguidores-de-milei-del-comercio-a-la-politica/)).

✅ To credit this academic work and the scraping code, it is recommended to cite: **SILVA, Ergon Cugler de Moraes. *TelegramScrap: A comprehensive tool for scraping Telegram data*. (Feb) 2023. Available at: [https://github.com/ergoncugler/web-scraping-telegram/](https://github.com/ergoncugler/web-scraping-telegram/).**

In [ ]:
# @title **1. [ Required ] Set up your credentials once** { display-mode: "form" }

# @markdown Here, you need to input your credentials: `username`, `phone`, `api_id`, and `api_hash`. Your `api_id` and `api_hash` can only be generated from [Telegram's app creation page](https://my.telegram.org/apps). Once your credentials are set up, you won't need to update them again. Just click "Run" to proceed.

# Install the Telethon library for Telegram API interactions
!pip install -q telethon
!pip install -q boto3 requests-aws4auth

# Initial imports
from datetime import datetime, timezone
import pandas as pd
import time
import json
import re

# Telegram imports
from telethon.sync import TelegramClient

# Google Colab imports (optional for AWS version)
try:
    from google.colab import files
    COLAB_MODE = True
except ImportError:
    COLAB_MODE = False
    print("Running in non-Colab environment")

# === AWS INTEGRATION ===
# Import AWS functions from external file
try:
    from aws_integration import get_aws_integration, send_to_kinesis, send_batch_to_kinesis
    aws = get_aws_integration()
    AWS_AVAILABLE = aws.get_status()['aws_ready']
    print(f"🔥 AWS Status: {'✅ ATIVO' if AWS_AVAILABLE else '❌ INATIVO'}")
except ImportError:
    print("⚠️ AWS integration não disponível")
    AWS_AVAILABLE = False

# === CARREGAR CREDENCIAIS DAS VARIÁVEIS DE AMBIENTE ===
import sys
import os
sys.path.append('./src')

# Pegar diretamente das variáveis de ambiente (já configuradas na VM)
username = os.getenv('TELEGRAM_USERNAME', 'ergoncugler_cinco')
phone = os.getenv('TELEGRAM_PHONE', '+5513992053110')  
api_id = os.getenv('TELEGRAM_API_ID', '25314141')
api_hash = os.getenv('TELEGRAM_API_HASH', '42ff390ccfc06424340dbdd37faabc35')

print("✅ Credenciais carregadas das variáveis de ambiente:")
print(f"Username: {username}")
print(f"Phone: {phone}")
print(f"API ID: {api_id}")
print(f"API Hash: {api_hash[:8]}...")


[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Running in non-Colab environment
🔧 Inicializando integração AWS...
✅ Kinesis Stream 'telegram-messages' está ATIVO
✅ CloudWatch habilitado
📈 CloudWatch: ✅ Habilitado
🚀 Sistema híbrido ativo: Jupyter + AWS
📡 Stream: telegram-messages
📈 CloudWatch: ✅
🔥 AWS Status: ✅ ATIVO
⚠️ Config loader não disponível - configure manualmente:


In [ ]:
# @title **2. [ Required ] Adjust every time you want to use it** { display-mode: "form" }

# @markdown In this section, you will define the parameters for scraping data from Telegram channels or groups. Specify the channels you want to scrape using the format `@ChannelName` or the full URL `https://t.me/ChannelName`. Do not use URLs starting with `https://web.telegram.org/`. Set the date range by defining the start and end day, month, and year. Choose an output file name for the scraped data. Optionally, set a search keyword if you need to filter messages by specific terms. Define the maximum number of messages to scrape and set a timeout in seconds.

# LOAD GROUPS FROM GOOGLE SHEETS (NEW AWS FEATURE)
try:
    from utils.google_sheets import GoogleSheetsLoader
    print("📊 Carregando grupos da planilha Google Sheets...")
    sheets_loader = GoogleSheetsLoader()
    groups_data = sheets_loader.load_groups()
    
    # Convert to format expected by notebook
    active_groups = [group['username'] for group in groups_data if group.get('active', True)]
    sheets_channels = ", ".join(active_groups[:10])  # Limit to first 10 for testing
    
    print(f"✅ {len(groups_data)} grupos carregados da planilha")
    print(f"🎯 Grupos ativos selecionados: {len(active_groups)}")
    print(f"📋 Primeiros grupos: {sheets_channels}")
    
except ImportError:
    print("⚠️ Google Sheets loader não disponível - usando grupos padrão")
    sheets_channels = ""
    groups_data = []

# Setup / change every time to define scraping parameters

# @markdown **2.1.** Here you put the name of the channel or group that you want to scrape, as an example, play: '@LulanoTelegram' or 'https://t.me/LulanoTelegram'. Do not use: 'https://web.telegram.org/a/#-1001249230829' or '-1001249230829'. **Just write the `channel names` always separated by commas (,):**
if sheets_channels:
    channels = sheets_channels # Load from Google Sheets
else:
    channels = "@LulanoTelegram, @jairbolsonarobrasil, @Other_Channel_Name" # @param {type:"string"}

channels = [channel.strip() for channel in channels.split(",")]

# @markdown **2.2.** Here you can select the `time window` you would like to extract data from the listed communities:
date_min = '2024-10-15' # @param {type:"date"}
date_max = '2025-01-15' # @param {type:"date"}

date_min = datetime.fromisoformat(date_min).replace(tzinfo=timezone.utc)
date_max = datetime.fromisoformat(date_max).replace(tzinfo=timezone.utc)

# @markdown **2.3.** Choose a `name` for the final file you want to download as output:
file_name = 'Test' # @param {type:"string"}

# @markdown **2.4.** `Keyword` to search, **leave empty if you want to extract all messages from the channel(s):**
key_search = '' # @param {type:"string"}

# @markdown **2.5.** **Maximum** `number of messages` to scrape (only use if you want a specific limit, otherwise leave a high number to scrape everything):
max_t_index = 1000000   # @param {type:"integer"}

# @markdown **2.6.** `Timeout in seconds` (never leave it longer than 6 hours, that is 21600 seconds, as Google Colab deactivates itself after that time):
time_limit = 21600 # @param {type:"integer"}

# @markdown **2.7.** Choose the format of the final file you want to download. If you are a first-time user, choose `Excel`. If you have advanced skills, you can use `Parquet`:
File = 'excel' # @param ["excel", "parquet"]

print(f"📺 Total de {len(channels)} grupos selecionados para scraping")

📊 Carregando grupos da planilha Google Sheets...
📊 Carregando grupos da planilha Google Sheets...
📊 DataFrame shape: (612, 17)
📊 Colunas encontradas: ['To Scrape', 'Url', 'Group', 'Project', 'Country', 'Format', 'Spectrum', 'Stance', 'Identity', 'Basis', 'Territory', 'Name', 'Description', 'Users', 'To Categorize', 'Chip', '2025-07']
✅ Carregados 204 grupos da planilha
✅ 204 grupos carregados da planilha
🎯 Grupos ativos selecionados: 204
📋 Primeiros grupos: @SputnikBrasil, @rt_brasil, @plenonews, @politicamente_incorreto_grupo_3, @brasilgrandeafrica, @rasgandooverbo, @Brasilverdadeiro, @terrabrasilnoticias1, @mblivre, @BOLSONAROGRUPO
📺 Total de 10 grupos selecionados para scraping


In [ ]:
# @title **3. [ Required ] Start Telegram scraping** { display-mode: "form" }

# @markdown **Attention:** During this step, Telegram may request a verification code. Please monitor your Telegram app and input the required information promptly. Rest assured, all data entered remains secure.

data = []  # List to store scraped data
t_index = 0  # Tracker for the number of messages processed
start_time = time.time()  # Record the start time for the scraping session

# Function to remove invalid XML characters from text
def remove_unsupported_characters(text):
    valid_xml_chars = (
        "[^\u0009\u000A\u000D\u0020-\uD7FF\uE000-\uFFFD"
        "\U00010000-\U0010FFFF]"
    )
    cleaned_text = re.sub(valid_xml_chars, '', text)
    return cleaned_text

# Function to format time in days, hours, minutes, and seconds
def format_time(seconds):
    days = seconds // 86400
    hours = (seconds % 86400) // 3600
    minutes = (seconds % 3600) // 60
    seconds = seconds % 60
    return f'{int(days):02}:{int(hours):02}:{int(minutes):02}:{int(seconds):02}'

# Function to print progress of the scraping process
def print_progress(t_index, message_id, start_time, max_t_index):
    elapsed_time = time.time() - start_time
    current_progress = t_index / (t_index + message_id) if (t_index + message_id) <= max_t_index else t_index / max_t_index
    percentage = current_progress * 100
    estimated_total_time = elapsed_time / current_progress
    remaining_time = estimated_total_time - elapsed_time

    elapsed_time_str = format_time(elapsed_time)
    remaining_time_str = format_time(remaining_time)

    print(f'Progress: {percentage:.2f}% | Elapsed Time: {elapsed_time_str} | Remaining Time: {remaining_time_str}')

# Function to classify messages (POL, CONSPIRA, NAZ) - CloudWatch PRO integration
def classify_message(content):
    """Classifica mensagem baseada no conteúdo"""
    if not content:
        return 'OTHER'
    
    content_lower = content.lower()
    
    # Keywords para classificação POL (política)
    pol_keywords = ['governo', 'presidente', 'política', 'eleição', 'voto', 'partido', 'deputado', 'senador', 'ministro', 'lula', 'bolsonaro', 'dilma', 'temer']
    
    # Keywords para classificação CONSPIRA (conspiração)
    conspira_keywords = ['conspiração', 'illuminati', 'maçonaria', 'nova ordem mundial', 'deep state', 'estado profundo', 'globalista', 'reptiliano', 'controle mental']
    
    # Keywords para classificação NAZ (nazismo/extremismo)
    naz_keywords = ['supremacista', 'raça superior', 'hitler', 'nazismo', 'facismo', '14/88', 'supremacia branca', 'nacionalismo branco']
    
    # Verificar POL
    if any(keyword in content_lower for keyword in pol_keywords):
        return 'POL'
    
    # Verificar CONSPIRA
    if any(keyword in content_lower for keyword in conspira_keywords):
        return 'CONSPIRA'
    
    # Verificar NAZ
    if any(keyword in content_lower for keyword in naz_keywords):
        return 'NAZ'
    
    return 'OTHER'

# Normalize File variable to avoid issues
File = re.sub(r'[^a-z]', '', File.lower())  # Converts to lowercase and removes non-alphabetic characters

# Scraping process
for channel in channels:
    if t_index >= max_t_index:
        break

    if time.time() - start_time > time_limit:
        break

    loop_start_time = time.time()

    try:
        c_index = 0
        async with TelegramClient(username, api_id, api_hash) as client:
            async for message in client.iter_messages(channel, search=key_search):
                try:
                    if date_min <= message.date <= date_max:

                        # Process comments of the message
                        comments_list = []
                        try:
                            async for comment_message in client.iter_messages(channel, reply_to=message.id):
                                comment_text = comment_message.text.replace("'", '"')

                                comment_media = 'True' if comment_message.media else 'False'

                                comment_emoji_string = ''
                                if comment_message.reactions:
                                    for reaction_count in comment_message.reactions.results:
                                        emoji = reaction_count.reaction.emoticon
                                        count = str(reaction_count.count)
                                        comment_emoji_string += emoji + " " + count + " "

                                comment_date_time = comment_message.date.strftime('%Y-%m-%d %H:%M:%S')

                                comments_list.append({
                                    'Type': 'comment',
                                    'Comment Group': channel,
                                    'Comment Author ID': comment_message.sender_id,
                                    'Comment Content': comment_text,
                                    'Comment Date': comment_date_time,
                                    'Comment Message ID': comment_message.id,
                                    'Comment Author': comment_message.post_author,
                                    'Comment Views': comment_message.views,
                                    'Comment Reactions': comment_emoji_string,
                                    'Comment Shares': comment_message.forwards,
                                    'Comment Media': comment_media,
                                    'Comment Url': f'https://t.me/{channel}/{message.id}?comment={comment_message.id}'.replace('@', ''),
                                })
                        except Exception as e:
                            comments_list = []
                            print(f'Error processing comments: {e}')

                        # Process the main message
                        media = 'True' if message.media else 'False'

                        emoji_string = ''
                        if message.reactions:
                            for reaction_count in message.reactions.results:
                                emoji = reaction_count.reaction.emoticon
                                count = str(reaction_count.count)
                                emoji_string += emoji + " " + count + " "

                        date_time = message.date.strftime('%Y-%m-%d %H:%M:%S')
                        cleaned_content = remove_unsupported_characters(message.text)
                        cleaned_comments_list = remove_unsupported_characters(json.dumps(comments_list))

                        # ===== CLOUDWATCH INTEGRATION =====
                        # Classificar mensagem para CloudWatch
                        message_category = classify_message(cleaned_content)
                        
                        # Atualizar contadores CloudWatch
                        if 'stats' in globals():
                            stats['total_messages'] += 1
                            if message_category == 'POL':
                                stats['pol_count'] += 1
                            elif message_category == 'CONSPIRA':
                                stats['conspira_count'] += 1
                            elif message_category == 'NAZ':
                                stats['naz_count'] += 1

                        message_data = {
                            'Type': 'text',
                            'Group': channel,
                            'Author ID': message.sender_id,
                            'Content': cleaned_content,
                            'Date': date_time,
                            'Message ID': message.id,
                            'Author': message.post_author,
                            'Views': message.views,
                            'Reactions': emoji_string,
                            'Shares': message.forwards,
                            'Media': media,
                            'Url': f'https://t.me/{channel}/{message.id}'.replace('@', ''),
                            'Comments List': cleaned_comments_list,
                            'Category': message_category,  # Adicionar categoria
                        }

                        data.append(message_data)  # Backup local

                        # === AWS HYBRID INTEGRATION ===
                        if AWS_AVAILABLE:
                            try:
                                aws_data = {
                                    'message_id': message.id,
                                    'group_name': channel,
                                    'content': cleaned_content,
                                    'classification': message_category,
                                    'timestamp': date_time,
                                    'author_id': message.sender_id,
                                    'views': message.views,
                                    'reactions': emoji_string,
                                    'shares': message.forwards,
                                    'media': media == 'True',
                                    'url': f'https://t.me/{channel}/{message.id}'.replace('@', ''),
                                }
                                success = send_to_kinesis(aws_data)
                                if success:
                                    print(f"✅ → AWS: {channel} | {message.id} | {message_category}")
                            except Exception as e:
                                print(f"⚠️ AWS: {e}")
                        # === END AWS INTEGRATION ===

                        c_index += 1
                        t_index += 1

                        # ===== ENVIAR MÉTRICAS CLOUDWATCH =====
                        if 'monitor' in globals() and monitor and hasattr(monitor, 'enabled') and monitor.enabled:
                            if t_index % 100 == 0:  # A cada 100 mensagens
                                stats['groups_count'] = len(channels)
                                monitor.send_scraping_stats(
                                    stats['total_messages'],
                                    stats['groups_count'], 
                                    stats['pol_count'],
                                    stats['conspira_count'],
                                    stats['naz_count']
                                )
                                print(f"📊 CloudWatch: {stats['total_messages']} mensagens")

                        # Print progress
                        print(f'{"-" * 80}')
                        print_progress(t_index, message.id, start_time, max_t_index)
                        current_max_id = min(c_index + message.id, max_t_index)
                        print(f'From {channel}: {c_index:05} contents of {current_max_id:05}')
                        print(f'Id: {message.id:05} / Date: {date_time}')
                        print(f'Total: {t_index:05} contents until now')
                        print(f'{"-" * 80}\n\n')

                        if t_index % 1000 == 0:
                            if File == 'parquet':
                                backup_filename = f'backup_{file_name}_until_{t_index:05}_{channel}_ID{message.id:07}.parquet'
                                pd.DataFrame(data).to_parquet(backup_filename, index=False)
                            elif File == 'excel':
                                backup_filename = f'backup_{file_name}_until_{t_index:05}_{channel}_ID{message.id:07}.xlsx'
                                pd.DataFrame(data).to_excel(backup_filename, index=False, engine='openpyxl')

                        if t_index >= max_t_index:
                            break

                        if time.time() - start_time > time_limit:
                            break

                    elif message.date < date_min:
                        break

                except Exception as e:
                    print(f'Error processing message: {e}')
                    # Enviar métrica de erro se CloudWatch estiver habilitado
                    if 'monitor' in globals() and monitor and hasattr(monitor, 'enabled') and monitor.enabled:
                        monitor.send_error_metric('MessageProcessing')

        print(f'\n\n##### {channel} was ok with {c_index:05} posts #####\n\n')

        df = pd.DataFrame(data)
        if File == 'parquet':
            partial_filename = f'complete_{channel}_in_{file_name}_until_{t_index:05}.parquet'
            df.to_parquet(partial_filename, index=False)
        elif File == 'excel':
            partial_filename = f'complete_{channel}_in_{file_name}_until_{t_index:05}.xlsx'
            df.to_excel(partial_filename, index=False, engine='openpyxl')
        # files.download(partial_filename)

    except Exception as e:
        print(f'{channel} error: {e}')
        # Enviar métrica de erro se CloudWatch estiver habilitado
        if 'monitor' in globals() and monitor and hasattr(monitor, 'enabled') and monitor.enabled:
            monitor.send_error_metric('ChannelProcessing')

    loop_end_time = time.time()
    loop_duration = loop_end_time - loop_start_time

    if loop_duration < 60:
        time.sleep(60 - loop_duration)

print(f'\n{"-" * 50}\n#Concluded! #{t_index:05} posts were scraped!\n{"-" * 50}')

# === CLOUDWATCH FINALIZAÇÃO ===
try:
    if 'monitor' in globals() and monitor and hasattr(monitor, 'enabled') and monitor.enabled and 'stats' in globals():
        print(f'\n🚀 Enviando métricas finais CloudWatch...')
        
        # Calcular tempo total de execução
        total_execution_time = time.time() - start_time
        
        # Atualizar contagem final de grupos
        stats['groups_count'] = len(channels)
        
        # Enviar métricas finais
        final_success = monitor.send_scraping_stats(
            stats['total_messages'],
            stats['groups_count'], 
            stats['pol_count'],
            stats['conspira_count'],
            stats['naz_count']
        )
        
        # Enviar métrica de duração
        if total_execution_time > 0:
            monitor.send_metric('ScrapingDuration', total_execution_time, 'Seconds')
            msgs_per_second = stats['total_messages'] / total_execution_time if total_execution_time > 0 else 0
            monitor.send_metric('MessagesPerSecond', msgs_per_second, 'Count/Second')
        
        if final_success:
            print(f'✅ Métricas enviadas para CloudWatch!')
        else:
            print(f'⚠️ CloudWatch: erro ao enviar métricas')
    else:
        print(f'\n⚠️ CloudWatch não habilitado - métricas não enviadas')
        
except Exception as e:
    print(f'\n❌ Erro ao processar métricas CloudWatch: {e}')

# === AWS BATCH FINAL (OPCIONAL) ===
if AWS_AVAILABLE and data:
    try:
        print(f'\n📤 Enviando batch final AWS: {len(data)} mensagens...')
        success_count, total_count = send_batch_to_kinesis(data)
        print(f'📡 Pipeline AWS: {success_count}/{total_count} enviado')
        if success_count == total_count:
            print(f'🎉 SCRAPING HÍBRIDO COMPLETO! (Jupyter + AWS)')
        else:
            print(f'⚠️ {total_count - success_count} mensagens falharam - dados salvos localmente')
    except Exception as e:
        print(f'❌ Erro batch AWS: {e}')

# Gerar arquivo final
df = pd.DataFrame(data)
if File == 'parquet':
    final_filename = f'FINAL_{file_name}_with_{t_index:05}.parquet'
    df.to_parquet(final_filename, index=False)
elif File == 'excel':
    final_filename = f'FINAL_{file_name}_with_{t_index:05}.xlsx'
    df.to_excel(final_filename, index=False, engine='openpyxl')

# Download do arquivo (apenas no Colab)
if COLAB_MODE:
    files.download(final_filename)
else:
    print(f'✅ Arquivo salvo como: {final_filename}')

# Mostrar resumo final com CloudWatch
print(f"""
🎉 SCRAPING COMPLETO COM CLOUDWATCH!

📊 Estatísticas finais:
- Total mensagens: {len(data):,}
- Grupos processados: {len(channels)}
- Tempo total: {(time.time() - start_time) / 60:.1f} minutos

🏛️ Classificação das mensagens:""")

if 'stats' in globals():
    total_classified = stats['pol_count'] + stats['conspira_count'] + stats['naz_count']
    if total_classified > 0:
        print(f"- POL (Política): {stats['pol_count']:,} ({stats['pol_count']/total_classified*100:.1f}%)")
        print(f"- CONSPIRA (Conspiração): {stats['conspira_count']:,} ({stats['conspira_count']/total_classified*100:.1f}%)")
        print(f"- NAZ (Extremismo): {stats['naz_count']:,} ({stats['naz_count']/total_classified*100:.1f}%)")
    else:
        print("- Aguardando classificação...")

if AWS_AVAILABLE:
    print(f"""
📡 Pipeline AWS:
- Status: ✅ HÍBRIDO ATIVO
- Stream: telegram-messages
- Dados enviados em tempo real
🚀 Status Pipeline: ✅ HÍBRIDO ATIVO""")
    
if CLOUDWATCH_ENABLED:
    print(f"🌐 Dashboard: https://console.aws.amazon.com/cloudwatch/home?region=us-east-1#dashboards:name=TelegramScrap-Dashboard")
    print(f"📈 Monitoramento CloudWatch ativo!")
else:
    print(f"⚠️ CloudWatch desabilitado - configure credenciais AWS")

print(f"""
✅ Arquivo salvo: {final_filename}
📋 Dados disponíveis para análise posterior!
""")

@SputnikBrasil error: invalid literal for int() with base 10: 'YOUR_API_ID'
@rt_brasil error: invalid literal for int() with base 10: 'YOUR_API_ID'
@plenonews error: invalid literal for int() with base 10: 'YOUR_API_ID'


In [ ]:
# @title **2.5. [ OPTIONAL ] CloudWatch Monitoring Ready** { display-mode: "form" }

# @markdown CloudWatch monitoring está configurado e pronto para uso quando AWS estiver ativo.

print("📊 Sistema preparado para CloudWatch")
print("💡 CloudWatch será habilitado automaticamente quando credenciais AWS estiverem configuradas")
print("🎯 Para ativar: export AWS_ACCESS_KEY_ID=... na EC2")

In [ ]:
# @title **2.6. [ NEW ] CloudWatch Monitoring** { display-mode: "form" }

# @markdown Configure CloudWatch monitoring para TelegramScrap.

# Initialize CloudWatch Monitor (versão simples)
try:
    from utils.monitoring import CloudWatchMonitor
    
    print("🚀 Inicializando CloudWatch Monitor...")
    monitor = CloudWatchMonitor()
    
    print(f"📈 CloudWatch: {'✅ Habilitado' if monitor.enabled else '⚠️ Desabilitado (sem credenciais AWS)'}")
    
    if monitor.enabled:
        print("📊 Dashboard: https://console.aws.amazon.com/cloudwatch/home?region=us-east-1#dashboards:name=TelegramScrap-Dashboard")
        print("📈 Métricas: AWS Console → CloudWatch → Metrics → TelegramScrap")
        
        # Inicializar contadores para estatísticas
        stats = {
            'total_messages': 0,
            'groups_count': 0, 
            'pol_count': 0,
            'conspira_count': 0,
            'naz_count': 0,
            'start_time': time.time()
        }
        print("✅ Contadores de estatísticas inicializados")
    else:
        print("💡 Configure credenciais AWS para habilitar CloudWatch")
        stats = {'total_messages': 0, 'groups_count': 0, 'pol_count': 0, 'conspira_count': 0, 'naz_count': 0}
    
    CLOUDWATCH_ENABLED = monitor.enabled
    
except ImportError:
    print("⚠️ CloudWatch Monitor não disponível")
    print("💡 Verifique se src/utils/monitoring.py existe")
    monitor = None
    CLOUDWATCH_ENABLED = False
    stats = {'total_messages': 0, 'groups_count': 0, 'pol_count': 0, 'conspira_count': 0, 'naz_count': 0}

print(f"🔥 CloudWatch Status: {'ATIVO' if CLOUDWATCH_ENABLED else 'INATIVO'}")

if CLOUDWATCH_ENABLED:
    print("\n✨ Funcionalidades ativas:")
    print("  📊 Métricas em tempo real")
    print("  🏛️ Classificação de mensagens (POL/CONSPIRA/NAZ)")
    print("  ⚡ Monitoramento automático")

In [ ]:
# @title **4. [ Bonus ] Reading your parquet file** { display-mode: "form" }

# @markdown If you want to read your generated file or convert from `.parquet` to another format, feel free to use it here.

# Import pandas for data manipulation
try:
    from google.colab import files
    COLAB_MODE = True
except ImportError:
    COLAB_MODE = False

import pandas as pd

# @markdown **4.1.** Set the Parquet file name here, including the .parquet extension:
filename = 'TYPE_HERE_YOUR_FILENAME.parquet'  # @param {type:"string"}
# Read the final Parquet file
df_parquet = pd.read_parquet(filename)

# Display the dataframe to the user (optional)
display(df_parquet)

# Convert the dataframe to Excel and set the new filename
excel_filename = filename.replace('.parquet', '.xlsx')

# Save the dataframe as an Excel file
df_parquet.to_excel(excel_filename, index=False)

# Download the Excel file (only in Colab)
if COLAB_MODE:
    files.download(excel_filename)
else:
    print(f"✅ Arquivo salvo como: {excel_filename}")
    print("📋 Em ambiente não-Colab, arquivo fica disponível localmente")